# Experiment: Case-001 Branch Review Response Capture Gate

Objective: smoke-test branch-review response capture with synthetic rows only.

Success criteria:
- private-material rows block public release;
- missing-domain rows route to private follow-up;
- branch rows emit branch-scope labels only, not treatment instructions;
- private-only fields are absent from public output.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

repo_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "scripts" / "capture_branch_review_response.py").exists()
)
sys.path.insert(0, str(repo_root / "scripts"))

import capture_branch_review_response as capture  # noqa: E402

capture.ALLOWED_RESPONSE_LABELS

## Plan

Use synthetic labels to verify the public projection. No real clinician text, patient record, doctor identity, hospital name, or contact detail is used.


In [ ]:
def synthetic_row(
    response_label: str,
    branch_label: str = "none",
    packet_domain: str = "none",
) -> dict[str, str]:
    """Return one synthetic private response row."""
    public_summary_label = capture.expected_public_label(response_label, branch_label)
    return {
        "response_label": response_label,
        "branch_label": branch_label,
        "packet_domain": packet_domain,
        "reviewer_role_label": "qualified_clinician",
        "private_response_ref": "synthetic-private-response-ref",
        "public_summary_label": public_summary_label,
        "notes_private": "synthetic private note",
    }


blocked_result = capture.capture_rows(
    [synthetic_row("release_blocked_private_material")]
)
assert blocked_result["current_label"] == capture.BLOCKED_LABEL
assert not capture.PRIVATE_ONLY_COLUMNS.intersection(blocked_result)

blocked_result

In [ ]:
followup_result = capture.capture_rows(
    [
        synthetic_row(
            "missing_packet_domain",
            packet_domain="iron_organ_chelation_status",
        )
    ]
)
assert followup_result["current_label"] == capture.FOLLOWUP_LABEL
assert followup_result["missing_domains"] == ["iron_organ_chelation_status"]
assert not capture.PRIVATE_ONLY_COLUMNS.intersection(followup_result)

followup_result

In [ ]:
branch_result = capture.capture_rows(
    [
        synthetic_row(
            "branch_in_scope_for_private_review",
            branch_label="autologous_gene_cell",
        )
    ]
)
assert branch_result["current_label"] == capture.CAPTURED_LABEL
assert branch_result["branch_scope_labels"] == [
    "branch_in_scope_for_private_review_autologous_gene_cell"
]
assert not capture.PRIVATE_ONLY_COLUMNS.intersection(branch_result)

branch_result

## Result

The synthetic gate keeps branch replies as public-safe labels. It does not parse raw medical text and does not create patient-specific action instructions.
